# E2 results (CarbonCast-faithful, production-based CI)

Trains and evaluates the **E2** model — the CarbonCast-faithful, production-based carbon-intensity baseline. E2 is the two-tier pipeline built in `carbon_forecast.models`: per-source Tier 1 ANNs feed a Tier 2 CNN-LSTM, with out-of-fold Tier 1 forecasts used for honest Tier 2 training (see `carboncast_faithful.py`).

This notebook imports the orchestrator; it never reimplements model logic. It loads the materialized `data/processed/{zone}.parquet` frames, so it is portable to Colab (upload the 5 small processed files; no API key needed for training).

In [ ]:
# --- Setup (local or Colab) ---
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # 1) Install the package, e.g. after cloning the private repo into /content:
    #    !pip -q install -e /content/carbon-intensity-forecast
    # 2) Mount Drive and point DATA_ROOT at the uploaded processed parquets.
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = "/content/drive/MyDrive/carbon-intensity-forecast/data"
else:
    sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
    DATA_ROOT = os.path.abspath(os.path.join("..", "data"))

os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from carbon_forecast.data import storage
from carbon_forecast.plotting import config as P
from carbon_forecast.models.carboncast_faithful import (
    train_e2, evaluate_ci, predict_with_truth, E2Config,
)
from carbon_forecast.models.tier1_source import Tier1Config
from carbon_forecast.models.tier2_cnnlstm import Tier2Config

P.apply_defaults()
print("Colab:", IN_COLAB, "| DATA_ROOT:", DATA_ROOT)

## Data and split

Locked split is Train 2021–2025, Validation 2026 Jan–Apr, Test A 2026 May, Test B 2026 Jun 8–21. 2026 history is not yet extracted, so the cell below uses an available **proxy split** (Train 2021–23, Val 2024, Test 2025). Switch the date strings once 2026 is pulled; nothing else changes.

In [ ]:
ZONES = ["BE", "FI", "SG", "US-MIDA-PJM", "US-NY-NYIS"]

def load_zone(zone):
    return storage.read_parquet(storage.processed_path(zone, DATA_ROOT))

# Proxy split (see note above). Once 2026 is extracted, use:
#   train=slice('2021','2025'), val=slice('2026-01','2026-04'), test=slice('2026-05','2026-05')
SPLIT = dict(train=slice("2021", "2023"), val=slice("2024", "2024"), test=slice("2025", "2025"))

be = load_zone("BE")
train, val, test = be.loc[SPLIT["train"]], be.loc[SPLIT["val"]], be.loc[SPLIT["test"]]
for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s} {part.index.min().date()} -> {part.index.max().date()}  ({len(part):,} h)")

## Train E2 (BE)

Full settings: 100-epoch cap with early stopping, stride 1. Out-of-fold fold models use a fixed 60-epoch budget (no early stopping). This is the first real training run — expect minutes on CPU, less on a Colab GPU.

In [ ]:
cfg = E2Config(
    tier1=Tier1Config(epochs=100, stride=1, patience=10),       # final source models
    tier1_fold=Tier1Config(epochs=60, stride=1),                # out-of-fold models
    tier2=Tier2Config(epochs=100, stride=1, patience=10),       # CNN-LSTM
)

t0 = time.time()
art = train_e2(train, val, "BE", cfg, verbose=1)
print(f"\ntrained BE E2 in {(time.time() - t0) / 60:.1f} min")

## Evaluate on the test fold

In [ ]:
metrics = evaluate_ci(art, test)
print("E2 — BE test (production-based CI):")
print(f"  MAPE : {metrics['mape_pct']:.2f} %   (primary)")
print(f"  MAE  : {metrics['mae']:.2f} gCO2eq/kWh")
print(f"  RMSE : {metrics['rmse']:.2f} gCO2eq/kWh")
print(f"  n    : {metrics['n_samples']:,} forecast origins")

## Per-horizon degradation curve

Error as a function of how far ahead we forecast (1–96 h). This is the Contribution-4 signature figure; the EM head-to-head later overlays on the 1–24 h band (academic-key forecast horizon).

In [ ]:
preds, y_true, origins = predict_with_truth(art, test)
ae = np.abs(preds - y_true)
mape_h = np.nanmean(ae / np.clip(np.abs(y_true), 1e-6, None), axis=0) * 100
hours = np.arange(1, preds.shape[1] + 1)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=hours, y=mape_h, mode="lines",
    line=dict(color=P.MODEL_PALETTE["carboncast_faithful"], width=P.LINE_WIDTH),
    name="E2 (CarbonCast-faithful)",
))
for h in [24, 48, 72]:
    fig.add_vline(x=h, line=dict(color="rgba(0,0,0,0.15)", width=1))
fig.update_xaxes(title="forecast horizon (hours ahead)")
fig.update_yaxes(title="MAPE (%)")
P.style_fig(fig, "E2 carbon-intensity error by forecast horizon",
            subtitle="BE, production-based CI, test 2025")
fig.show()

## All five zones

The cell below trains E2 across every zone and collects the headline metrics. This is the heavy run (≈ 6 Tier-1 trainings × ~sources × 5 zones + 5 Tier-2 fits) — a good candidate for the Colab Pro GPU. Uncomment to run.

In [ ]:
# results = {}
# for zone in ZONES:
#     df = load_zone(zone)
#     ztr, zva, zte = df.loc[SPLIT['train']], df.loc[SPLIT['val']], df.loc[SPLIT['test']]
#     t0 = time.time()
#     zart = train_e2(ztr, zva, zone, cfg, verbose=0)
#     results[zone] = {**evaluate_ci(zart, zte), 'minutes': (time.time()-t0)/60}
#     print(f"{zone:14s} MAPE={results[zone]['mape_pct']:.2f}%  ({results[zone]['minutes']:.1f} min)")
# pd.DataFrame(results).T